In [1]:
import pandas as pd

from tqdm import tqdm
import json
import os
from collections import defaultdict
from ast import literal_eval

import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import sys
# Add the directory to your path
sys.path.append('/home/sp3945/mod/ruler/')

from wordllama import WordLlama
wl = WordLlama.load()


from ruler.data.constants import URL_REPLACEMENT_TOKEN
from ruler.data.text_utils import markdown_to_text, replace_urls


In [2]:
MANUALLY_ANNOTATED = pd.read_csv("../data/interim/manually_annotated.csv", header=0)

In [3]:
annots = MANUALLY_ANNOTATED[MANUALLY_ANNOTATED['applied_rule_n']>0].set_index("ap_id").to_dict()['applied_rule_n']
len(annots.keys())

891

In [4]:
bad_reasons = ['lemmy.ml', 'instance', 'lemmy.world', 'hexbear']
manual_inspection = []
rows = []
with open("/storage/lemmymod/nonbinary_modlogs.jsonl", 'w+', encoding='utf8') as wfile:
    with open("/storage/lemmymod/modlogs_fuzzy_1.jsonl", "r") as jfile:
        for line in jfile:
            job = json.loads(line)
            if "user content mass removed" in job['reason'].lower(): continue
            job['number_of_rules'] = len(job['community']['rules']['rules'].keys())
            if job['number_of_rules'] <=1: continue
            if "instance" in job['reason'].lower(): continue
            if "lemmy" in job['reason'].lower(): continue
            if "hex" in job['reason'].lower(): continue
            if job['applied_rule_n']==-1:
                if job['ap_id'] in annots:
                    job['applied_rule_n'] = int(annots[job['ap_id']])
                    job['applied_rule_text'] = job['community']['rules']['rules'][str(int(annots[job['ap_id']]))]

            
            if job['applied_rule_n']>0:
            #reason = job['reason']
            #rules = job['community']['rules']
            #fuzzreaon = fuzzymatch_rules(wl, reason, rules)
                rows.append(job)
                wfile.write(json.dumps(job, sort_keys=True, ensure_ascii=False) + '\n')
            # if job['applied_rule_n']==-1:
            #     tempd = {}
            #     tempd['ap_id'] = job['ap_id']
            #     tempd['reason'] = job['reason']
            #     tempd['rules'] = job['community']['rules']
            #     tempd['applied_rule_n'] = job['applied_rule_n']
            #     manual_inspection.append(tempd)


In [ ]:
data = pd.DataFrame(rows)

In [ ]:
#data.to_csv("./manual_inspection.csv")

In [ ]:
from collections import defaultdict, Counter
print(Counter(data['applied_rule_n'].tolist()))

In [ ]:
len(data)-22912

In [ ]:
data.sample(5)

In [ ]:
data['fuzzymatch'] = data.apply(lambda row: fuzzymatch_rules(wl, row['reason'], row['community']['rules']) if row['applied_rule_n']!=-1 else None, axis=1)

In [ ]:
fuzzspect = data.dropna(subset=['fuzzymatch'])
fuzzspect[fuzzspect['fuzzymatch']!=-1].to_csv("./fuzzmatch_sample_07.csv")

In [ ]:
fuzzspect[fuzzspect['fuzzymatch']!=-1]

In [ ]:
from collections import defaultdict,Counter
print(Counter(data['applied_rule_n'].tolist()))

In [ ]:
len(data) - 24305

In [ ]:
print(list(data))

In [ ]:
data['number_of_rules'] = data['community'].apply(lambda x: len(x['rules']['rules'].keys()))

In [ ]:
print(Counter(data['number_of_rules'].tolist()))

In [ ]:
print(Counter(data[data['number_of_rules']>1]['applied_rule_n'].tolist()))

In [ ]:
len(data[data['number_of_rules']>1])

In [ ]:
33590-24511